# Amendment 30: the three CPU checks without clipping

All sessions: **CPU + High-RAM**, notebook **saved in your Drive** (File → Save a copy in Drive), the same commit SHA in cell 1.

| Session | ROLE | What it does | Cells |
|---|---|---|---|
| 1 | `F1` | learning-rate check: 349 runs without clipping, then the selection without clipping | 1–7, then 8, 9 |
| 2 | `F2` | squash check: 100 runs without clipping, then the squash analysis without clipping (waits for F3's 9 runs) | 1–7, then 10, 11 |
| 3 | `F3` | wiring check: 9 runs without clipping, then the wiring analysis without clipping | 1–7, then 12, 13 |
| any, at the end | — | verdict counts | 14 |

**Rules:** every cell must end with `OK`; if not, stop and send the output. Results go to Drive; finished runs are skipped on a restart (run cells 1–7 again, then the cell you were on). No cell shows an estimate while runs are still going.

In [ ]:
# 1. SETTINGS. Edit these two lines only.
EXPECTED_SHA = "PASTE_THE_FULL_SHA_OF_THE_AMENDMENT_30_COMMIT"
ROLE = "F1"            # "F1", "F2" or "F3"

BACKUP = "/content/drive/MyDrive/QML-backup-run"
EXP31  = BACKUP + "/exp31"                 # Amendment 26 records (read only)
OUT    = BACKUP + "/cpu_reruns"            # Amendments 27-29 outputs; this notebook writes CU_* here
REPO, ART, ENV = "/content/repo", "/content/art", "/content/env"
PY = ENV + "/bin/python"
SHA26, SHA29 = "58852d84", "56a09feb"      # commits that recorded 31_entangling_range, and 32_squash + 34_lr_inherited
DATASETS = ["breastmnist", "pneumoniamnist", "bloodmnist", "pathmnist"]
REGIMES  = ["5", "10", "20", "50", "100"]
T28 = T34 = 2                              # threads per process of the C_32 and C_34 re-executions
TUNE_SEEDS = [90001, 90002, 90003, 90004, 90005]
EXPECT_CU = {"31_entangling_range": 9, "32_squash": 100, "34_lr_inherited": 349}   # Amendment 30

import os, subprocess, json, glob, time, shutil, re
assert ROLE in ("F1", "F2", "F3"), ROLE
os.environ.update({"QML_ARTIFACT_ROOT": ART, "MPLBACKEND": "Agg", "CUDA_VISIBLE_DEVICES": ""})

def sh(cmd, log=None, check=True, env=None):
    """Run a bash command from the repo, print its output, append it to a log on Drive."""
    e = dict(os.environ, **(env or {}))
    p = subprocess.Popen(["bash", "-lc", cmd], cwd=REPO if os.path.isdir(REPO) else "/content",
                         stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, env=e)
    lines = []
    for line in p.stdout:
        print(line, end="")
        lines.append(line)
    rc = p.wait()
    if log:
        os.makedirs(OUT + "/logs", exist_ok=True)
        with open(f"{OUT}/logs/{log}", "a") as f:
            f.write(f"\n$ {cmd}\n" + "".join(lines) + f"exit code {rc}\n")
    if check and rc != 0:
        raise RuntimeError(f"FAILED (exit {rc}): {cmd}")
    return "".join(lines)

def launch(cmds, threads, tag):
    """Start commands side by side in the background, each with `threads` threads. Logs on Drive."""
    env = dict(os.environ, OMP_NUM_THREADS=str(threads), MKL_NUM_THREADS=str(threads))
    procs = {}
    for name, cmd in cmds.items():
        log = f"{OUT}/logs/{tag}_{name}.txt"
        procs[name] = subprocess.Popen(["bash", "-lc", f"{cmd} >> {log} 2>&1"], cwd=REPO,
                                       env=env, start_new_session=True)
        print(f"started {name} (pid {procs[name].pid}, {threads} threads), log: {log}")
    return procs

def wait(procs, every=900, show=None):
    """Wait for background processes; print progress every 15 minutes."""
    while any(p.poll() is None for p in procs.values()):
        if show: show()
        time.sleep(every)
    if show: show()
    codes = {n: p.returncode for n, p in procs.items()}
    print("exit codes:", codes)
    assert all(c == 0 for c in codes.values()), "a process failed: see its log on Drive"

def verdicts(target):
    """Count replay verdicts in a target namespace on Drive."""
    from collections import Counter
    c = Counter()
    for fn in glob.glob(f"{OUT}/shards/{target}/*.json"):
        try:
            v = json.load(open(fn)).get("replay_check", {}).get("verdict", "?")
        except Exception:
            v = "unreadable"
        c[v.split(" (")[0]] += 1
    return dict(c)

def needs_list(source, path):
    """Shard names of the runs whose C_ re-execution shows a step above 20 (Amendment 30)."""
    out = sh(f"{PY} scripts/build_unclipped_view.py --list {source}")
    names = [l.strip() for l in out.splitlines() if l.strip().endswith(".json")]
    assert len(names) == EXPECT_CU[source], f"{source}: {len(names)} runs to re-execute, expected {EXPECT_CU[source]}"
    open(path, "w").write("\n".join(names) + "\n")
    return names

def run_unclipped(source, sha, names, threads, n_procs, tag):
    """The first run alone (it creates the code worktree and must be PREFIX IDENTICAL), then the rest in
    n_procs processes. Progress and a time estimate from the runs done so far every 15 minutes."""
    target = "CU_" + source
    base = f"{PY} -u scripts/replay.py --source {source} --clip inf --code-at {sha}"
    open(f"/content/{tag}_first.txt", "w").write(names[0] + "\n")
    sh(f"OMP_NUM_THREADS={threads} MKL_NUM_THREADS={threads} {base} --from-list /content/{tag}_first.txt",
       log=f"{tag}_first.txt")
    v = verdicts(target)
    print("first run:", v)
    assert v and all(k.startswith("PREFIX IDENTICAL") for k in v), "first unclipped run not PREFIX IDENTICAL: stop, send the output"
    rest = names[1:]
    chunks = {f"p{i}": rest[i::n_procs] for i in range(n_procs) if rest[i::n_procs]}
    for nm, ch in chunks.items():
        open(f"/content/{tag}_{nm}.txt", "w").write("\n".join(ch) + "\n")
    t0, n0 = time.time(), sum(verdicts(target).values())
    def show():
        n = sum(verdicts(target).values())
        dt = (time.time() - t0) / 3600
        eta = f", about {dt * (len(names) - n) / (n - n0):.1f} h left at the rate so far" if n > n0 and dt > 0.1 else ""
        print(time.strftime("%H:%M"), f"{target}: {n} of {len(names)}{eta}", verdicts(target), flush=True)
    wait(launch({nm: f"{base} --from-list /content/{tag}_{nm}.txt" for nm in chunks}, threads, tag), show=show)
    v = verdicts(target)
    print(f"{target} verdicts:", v)
    assert sum(v.values()) == len(names), f"{target}: {sum(v.values())} of {len(names)} done"
    assert all(k.startswith("PREFIX IDENTICAL") for k in v), f"{target}: a run is not PREFIX IDENTICAL: send this output"

print("ROLE", ROLE, "OK")

In [ ]:
# 2. MOUNT DRIVE and check the inputs exist.
from google.colab import drive
drive.mount("/content/drive")
fc = sorted(glob.glob(BACKUP + "/feature_cache_*.tgz"))
assert fc, "feature_cache_*.tgz not found"
counts = {"31_entangling_range (exp31)": len(glob.glob(EXP31 + "/shards/31_entangling_range/*.json")),
          "C_31_entangling_range": len(glob.glob(OUT + "/shards/C_31_entangling_range/*.json")),
          "32_squash": len(glob.glob(OUT + "/shards/32_squash/*.json")),
          "C_32_squash": len(glob.glob(OUT + "/shards/C_32_squash/*.json")),
          "34_lr_inherited": len(glob.glob(OUT + "/shards/34_lr_inherited/*.json")),
          "C_34_lr_inherited": len(glob.glob(OUT + "/shards/C_34_lr_inherited/*.json"))}
print(counts)
assert list(counts.values()) == [2400, 2400, 1600, 1600, 3960, 3960], "inputs incomplete: send this output"
for f in ("squash_check.json", "lr_inherited.json"):
    assert os.path.exists(f"{OUT}/{f}"), f"{OUT}/{f} missing"
print("OK")

In [ ]:
# 3. CLONE THE REPO at the pushed commit; check it is clean and carries Amendment 30.
if not os.path.isdir(REPO):
    sh(f"git clone -b feature/journal-expansion https://github.com/anishfelixm/quantum-classical-expressivity {REPO}")
sh(f"git fetch origin && git checkout --detach {EXPECTED_SHA}")
assert sh("git rev-parse HEAD").strip() == EXPECTED_SHA
assert sh("git status --porcelain").strip() == "", "the clone has local changes"
assert "## Amendment 30" in open(REPO + "/docs/analysis_plan.md").read(), "Amendment 30 missing at this commit"
assert os.path.exists(REPO + "/scripts/build_unclipped_view.py")
for f in ("analyse_squash.py", "analyse_lr_inherited.py", "analyse_entangling_range.py"):
    assert "--unclipped" in open(f"{REPO}/scripts/{f}").read(), f + " lacks --unclipped"
print("OK")

In [ ]:
# 4. PYTHON ENVIRONMENT: Python 3.10 with requirements.lock.txt (about 5 minutes).
if not os.path.exists(PY):
    sh("curl -Ls https://micro.mamba.pm/api/micromamba/linux-64/latest | tar -xj -C /content bin/micromamba")
    sh(f"/content/bin/micromamba create -y -q -p {ENV} -c conda-forge python=3.10 pip")
    sh(f"{PY} -m pip install -q -r requirements.lock.txt pytest --extra-index-url https://download.pytorch.org/whl/cu118")
sh("apt-get -qq install -y pigz > /dev/null", check=False)
sh(f"{PY} -c \"import sys, torch; from importlib.metadata import version as v; print('python', sys.version.split()[0], '| torch', torch.__version__, '| pennylane', v('pennylane'), '| numpy', v('numpy'), '| default threads', torch.get_num_threads())\"",
   log="environment_a30.txt")
print("OK")

In [ ]:
# 5. DEVICE. CPU only; record this session (cores, processor, role).
info = sh(f"{PY} -c \"import sys; sys.path.insert(0,'src'); import config; print(config.DEVICE.type)\"").split()
assert info[-1] == "cpu", "Use the CPU runtime (Runtime -> Change runtime type -> CPU, High-RAM)."
cpu = sh("lscpu | grep 'Model name' | sed 's/.*: *//'", check=False).strip()
cores = os.cpu_count()
with open(OUT + "/logs/sessions.txt", "a") as f:
    f.write(f"{time.strftime('%Y-%m-%d %H:%M')} role={ROLE} cpu='{cpu}' cores={cores} sha={EXPECTED_SHA[:8]}\n")
print(f"cpu='{cpu}'  cores={cores}")
print("OK")

In [ ]:
# 6. GATES: parity and circuit-wiring tests.
sh(f"{PY} -m pytest -q tests/test_parity.py tests/test_entangling_ranges.py", log=f"gates_{ROLE}.txt")
print("OK")

In [ ]:
# 7. ARTIFACT FOLDER: links to Drive, the restored feature cache, the recorded analysis results.
def link(sub, ns, target_dir):
    os.makedirs(f"{ART}/{sub}", exist_ok=True)
    os.makedirs(target_dir, exist_ok=True)
    p = f"{ART}/{sub}/{ns}"
    if os.path.islink(p):
        os.unlink(p)
    elif os.path.isdir(p):
        shutil.rmtree(p)
    os.symlink(target_dir, p)

for sub in ("shards", "predictions"):
    link(sub, "31_entangling_range", f"{EXP31}/{sub}/31_entangling_range")
for ns in ("C_31_entangling_range", "32_squash", "C_32_squash", "34_lr_inherited", "C_34_lr_inherited",
           "CU_31_entangling_range", "CU_32_squash", "CU_34_lr_inherited"):
    for sub in ("shards", "predictions", "step_norms", "weights"):
        link(sub, ns, f"{OUT}/{sub}/{ns}")

# the 10 October feature cache (config.py creates an empty folder on import, so test for the link)
FC = f"{ART}/feature_cache"
if not os.path.islink(FC):
    if os.path.isdir(FC):
        shutil.rmtree(FC)
    pts = glob.glob("/content/fc_extract/**/*__r*__s*.pt", recursive=True)
    if not pts:
        os.makedirs("/content/fc_extract", exist_ok=True)
        sh(f"tar -I pigz -xf '{fc[-1]}' -C /content/fc_extract")
        pts = glob.glob("/content/fc_extract/**/*__r*__s*.pt", recursive=True)
    os.symlink(os.path.dirname(pts[0]), FC)
assert os.path.realpath(FC).startswith("/content/fc_extract")
seeds_all = json.loads(sh(f"{PY} -c \"import sys, json; sys.path.insert(0,'src'); import config; "
                          f"print(json.dumps(config.CONFIRMATORY_SEEDS))\"").strip().splitlines()[-1])
missing = [(d, r, s) for d in DATASETS for r in REGIMES for s in seeds_all
           if not os.path.exists(f"{FC}/{d}__r{r}__s{s}.pt")]
assert not missing, f"{len(missing)} cache files missing; stop and send this output"

# the tuning seeds' caches: the very files the learning-rate runs used, saved on Drive by session C.
# Never built here (a new build could differ, and a run that builds its own cache starts differently).
if ROLE == "F1":
    need = [f"{d}__r{r}__s{s}.pt" for d in DATASETS for r in ("5", "20", "100") for s in TUNE_SEEDS]
    for f in need:
        if not os.path.exists(f"{FC}/{f}") and os.path.exists(f"{OUT}/tune_cache/{f}"):
            shutil.copy(f"{OUT}/tune_cache/{f}", f"{FC}/{f}")
    miss = [f for f in need if not os.path.exists(f"{FC}/{f}")]
    assert not miss, f"{len(miss)} tuning cache files missing, e.g. {miss[:3]}: send this output"
    print(f"tuning-seed cache files present: {len(need)} of {len(need)}")

# the recorded results, for the side-by-side print of each analysis
shutil.copy(f"{OUT}/squash_check.json", f"{ART}/squash_check.json")
shutil.copy(f"{OUT}/lr_inherited.json", f"{ART}/lr_inherited.json")
er = [p for p in (f"{EXP31}/entangling_range.json", f"{OUT}/entangling_range.json") if os.path.exists(p)]
if er:
    shutil.copy(er[0], f"{ART}/entangling_range.json")
print("recorded wiring results:", er[0] if er else "NOT FOUND (cell 13 then also recomputes the recorded side)")
print("cache files present: 800 of 800")
print("OK")

In [ ]:
# 8. ROLE F1 - AMENDMENT 30, BLOCK C: the 349 learning-rate runs in which clipping acted, without clipping.
#    2 threads per process, as their clipped re-execution.
assert ROLE == "F1"
names = needs_list("34_lr_inherited", "/content/cu34_list.txt")
run_unclipped("34_lr_inherited", SHA29, names, T34, max(1, min(4, cores // T34)), "replay34u")
print("OK")

In [ ]:
# 9. ROLE F1 - AMENDMENT 30 P2: the learning-rate selection without clipping (seconds).
assert ROLE == "F1"
sh(f"{PY} -u scripts/build_unclipped_view.py --build 34_lr_inherited", log="a30_views.txt")
sh(f"{PY} -u scripts/analyse_lr_inherited.py --unclipped", log="stats_lr_inherited_unclipped.txt")
shutil.copy(f"{ART}/lr_inherited_unclipped.json", f"{OUT}/lr_inherited_unclipped.json")
print("OK")

In [ ]:
# 10. ROLE F2 - AMENDMENT 30, BLOCK B: the 100 squash runs in which clipping acted, without clipping.
assert ROLE == "F2"
names = needs_list("32_squash", "/content/cu32_list.txt")
run_unclipped("32_squash", SHA29, names, T28, max(1, min(4, cores // T28)), "replay32u")
print("OK")

In [ ]:
# 11. ROLE F2 - AMENDMENT 30 P3: the squash analysis without clipping (B = 2000, about 2 hours).
#     It needs F3's 9 wiring runs too (2 of them are VQC runs used here); it waits for them.
assert ROLE == "F2"
while sum(verdicts("CU_31_entangling_range").values()) < EXPECT_CU["31_entangling_range"]:
    print(time.strftime("%H:%M"), "waiting for F3: CU_31", verdicts("CU_31_entangling_range"), flush=True)
    time.sleep(300)
sh(f"{PY} -u scripts/build_unclipped_view.py --build 31_entangling_range 32_squash", log="a30_views.txt")
out = sh(f"{PY} -u scripts/analyse_squash.py --unclipped --workers {max(1, cores - 1)}",
         log="stats_squash_unclipped.txt")
assert "SEED-LEVEL FALLBACK IN USE" not in out, "STOP: seed-level fallback in use. Do not use these numbers."
shutil.copy(f"{ART}/squash_check_unclipped.json", f"{OUT}/squash_check_unclipped.json")
print("OK")

In [ ]:
# 12. ROLE F3 - AMENDMENT 30, BLOCK A: the 9 wiring runs in which clipping acted, without clipping.
#     Each with the threads of the session that recorded it: max(1, cores // 3), cores from exp31/logs/sessions.txt.
assert ROLE == "F3"
names = needs_list("31_entangling_range", "/content/cu31_list.txt")
for k in (0, 1):
    my_seeds = set(seeds_all[k::2])
    mine = [n for n in names if int(re.search(r"__seed(\d+)", n).group(1)) in my_seeds]
    if not mine:
        continue
    lines = [l for l in open(EXP31 + "/logs/sessions.txt") if f"session {k}/" in l and f"sha={SHA26}" in l]
    rec_cores = sorted({int(re.search(r"cores=(\d+)", l).group(1)) for l in lines})
    assert len(rec_cores) == 1, f"session {k}: core counts {rec_cores}: send this output"
    threads = max(1, rec_cores[0] // 3)
    print(f"original session {k}: {len(mine)} runs, cores={rec_cores[0]} -> {threads} threads")
    open(f"/content/cu31_s{k}.txt", "w").write("\n".join(mine) + "\n")
    sh(f"OMP_NUM_THREADS={threads} MKL_NUM_THREADS={threads} {PY} -u scripts/replay.py --source 31_entangling_range "
       f"--clip inf --code-at {SHA26} --from-list /content/cu31_s{k}.txt", log=f"replay31u_s{k}.txt")
v = verdicts("CU_31_entangling_range")
print("CU_31_entangling_range verdicts:", v)
assert sum(v.values()) == 9 and all(x.startswith("PREFIX IDENTICAL") for x in v), "send this output"
print("OK")

In [ ]:
# 13. ROLE F3 - AMENDMENT 30 P4: the wiring analysis without clipping (B = 2000, about 1.5 hours;
#     about 3 hours if the recorded results were not found in cell 7, because they are recomputed first).
assert ROLE == "F3"
sh(f"{PY} -u scripts/build_unclipped_view.py --build 31_entangling_range", log="a30_views.txt")
w = max(1, cores - 1)
if not os.path.exists(f"{ART}/entangling_range.json"):
    out = sh(f"{PY} -u scripts/analyse_entangling_range.py --workers {w}", log="stats_entangling_recorded_a30.txt")
    assert "SEED-LEVEL FALLBACK IN USE" not in out, "STOP: seed-level fallback in use."
out = sh(f"{PY} -u scripts/analyse_entangling_range.py --unclipped --workers {w}",
         log="stats_entangling_unclipped.txt")
assert "SEED-LEVEL FALLBACK IN USE" not in out, "STOP: seed-level fallback in use. Do not use these numbers."
shutil.copy(f"{ART}/entangling_range_unclipped.json", f"{OUT}/entangling_range_unclipped.json")
print("OK")

In [ ]:
# 14. ANY SESSION, AT THE END: verdict counts of the three unclipped blocks.
for t in ("CU_31_entangling_range", "CU_32_squash", "CU_34_lr_inherited"):
    print(f"{t:24s} {verdicts(t)}")
print("Expected (Amendment 30 P1): CU_31 PREFIX IDENTICAL 9; CU_32 PREFIX IDENTICAL 100; CU_34 PREFIX IDENTICAL 349.")
print("OK")